# Modelagem de Potencial de Consumo Municipal

## Objetivo

Desenvolver um indicador proprietário capaz de priorizar municípios brasileiros segundo seu potencial de consumo.

## Hipótese de Negócio

Mercados consumidores não devem ser avaliados apenas pelo tamanho populacional ou apenas pela renda disponível.

A combinação entre escala econômica e capacidade de consumo tende a representar melhor o potencial comercial de uma região.

## Variáveis Utilizadas

* PIB Municipal
* PIB per capita
* População Residente

## Aplicação de Negócio

O modelo pode apoiar decisões relacionadas a:

* Expansão comercial
* Definição de territórios de vendas
* Inteligência de mercado
* Planejamento de campanhas
* Priorização de investimentos

## Resultado Esperado

Ranking nacional de municípios segundo potencial estimado de consumo.


In [ ]:
# — Carregamento da Base Econômica Municipal ──────────────────────────────
# COMO funciona: lê a base processada criada no Notebook 02.
# ONDE é usado: será a base principal para receber os dados do CAGED.
# POR QUE esta escolha: evita recalcular todo o pipeline do IBGE.
# QUANDO rodar: primeira célula do notebook de integração CAGED.
# O QUE produz: DataFrame df_municipios contendo PIB, população e PIB per capita.

import pandas as pd

caminho_arquivo = (
    "dados/processados/"
    "base_municipios_pib_per_capita.csv"
)

df_municipios = pd.read_csv(
    caminho_arquivo,
    encoding="utf-8-sig"
)

print("Quantidade de municípios:")
print(len(df_municipios))

print("\nColunas disponíveis:")
print(df_municipios.columns.tolist())

Quantidade de municípios:
5570

Colunas disponíveis:
['NC', 'NN', 'MC', 'MN', 'pib_municipal', 'codigo_municipio', 'municipio', 'D2C', 'D2N', 'D3C', 'ano', 'populacao_residente', 'pib_per_capita', 'faixa_pib_per_capita']


In [ ]:
# — Criação do Score de Potencial de Consumo ──────────────────────────────
# COMO funciona: combina tamanho econômico e riqueza por habitante.
# ONDE é usado: priorização de mercados para expansão comercial.
# POR QUE esta escolha: municípios muito ricos nem sempre possuem escala,
# enquanto municípios muito grandes nem sempre possuem alto poder de compra.
# QUANDO rodar: após carregar a base processada.
# O QUE produz: coluna score_potencial_consumo.

import numpy as np

df_municipios["score_pib"] = (
    df_municipios["pib_municipal"]
    / df_municipios["pib_municipal"].max()
)

df_municipios["score_pib_per_capita"] = (
    df_municipios["pib_per_capita"]
    / df_municipios["pib_per_capita"].max()
)

df_municipios["score_potencial_consumo"] = (
    (
        df_municipios["score_pib"] * 0.6
    )
    +
    (
        df_municipios["score_pib_per_capita"] * 0.4
    )
)

df_municipios["score_potencial_consumo"] = (
    df_municipios["score_potencial_consumo"]
    .round(4)
)

print("Score criado com sucesso.")

Score criado com sucesso.


In [ ]:
# — Ranking de Potencial de Consumo ──────────────────────────────
# COMO funciona: ordena os municípios pelo score criado.
# ONDE é usado: identificação dos mercados prioritários.
# POR QUE esta escolha: permite validar se o score produz resultados coerentes.
# QUANDO rodar: após criar a coluna score_potencial_consumo.
# O QUE produz: ranking dos 20 municípios com maior potencial de consumo.

df_top_20_consumo = (
    df_municipios[
        [
            "municipio",
            "pib_municipal",
            "populacao_residente",
            "pib_per_capita",
            "score_potencial_consumo"
        ]
    ]
    .sort_values(
        by="score_potencial_consumo",
        ascending=False
    )
    .head(20)
)

display(df_top_20_consumo)

,municipio,pib_municipal,populacao_residente,pib_per_capita,score_potencial_consumo
3829,São Paulo - SP,945946483,11451999,82.600992,0.6392
3215,Maricá - RJ,158394291,197277,802.902979,0.4819
3255,Saquarema - RJ,75409090,89559,842.004600,0.4478
3499,Ilhabela - SP,24913501,34934,713.159129,0.3546
3677,Paulínia - SP,70496584,110537,637.764586,0.3477
2183,São Francisco do Conde - BA,25361789,38733,654.785041,0.3271
3155,Presidente Kennedy - ES,8179738,13696,597.235543,0.2889
3242,Rio de Janeiro - RJ,380381515,6211223,61.241001,0.2704
5569,Brasília - DF,328789563,2817381,116.700426,0.2640
5299,Santa Rita do Trivelato - MT,1409325,3276,430.196886,0.2053


## Insight de Negócio — Potencial de Consumo Municipal

O score revelou que o maior potencial de consumo não está necessariamente concentrado apenas nos maiores centros urbanos do país.

Embora São Paulo apareça na liderança devido à sua enorme escala econômica, municípios como Maricá (RJ), Saquarema (RJ), Paulínia (SP) e Ilhabela (SP) alcançam posições de destaque por combinarem elevada geração de riqueza com baixa densidade populacional.

Esse comportamento sugere a existência de mercados altamente atrativos para produtos e serviços de maior valor agregado, onde o poder econômico disponível por habitante é significativamente superior à média nacional.

Sob uma ótica de inteligência comercial, uma estratégia baseada apenas em tamanho populacional tenderia a ignorar municípios com elevado potencial de consumo. O score construído permite identificar esses mercados ocultos e priorizar regiões onde a probabilidade de conversão para produtos premium pode ser superior à observada em grandes centros urbanos.

Hipótese para validação futura:

Municípios com elevado PIB per capita e população intermediária podem apresentar maior propensão ao consumo de produtos financeiros, seguros, consórcios, energia solar, imóveis e bens de maior ticket médio.

In [ ]:
# — Comparação de Pesos do Score ──────────────────────────────
# COMO funciona: cria uma segunda versão do score com pesos diferentes.
# ONDE é usado: validação da modelagem de potencial de consumo.
# POR QUE esta escolha: permite verificar se o ranking está favorecendo
# excessivamente municípios pequenos com PIB per capita muito elevado.
# QUANDO rodar: após a criação do score_potencial_consumo.
# O QUE produz: coluna score_potencial_consumo_ajustado.

df_municipios["score_potencial_consumo_ajustado"] = (
    (
        df_municipios["score_pib"] * 0.8
    )
    +
    (
        df_municipios["score_pib_per_capita"] * 0.2
    )
)

df_municipios["score_potencial_consumo_ajustado"] = (
    df_municipios["score_potencial_consumo_ajustado"]
    .round(4)
)

print("Novo score criado com sucesso.")

Novo score criado com sucesso.


In [ ]:
# — Comparação dos Rankings ──────────────────────────────
# COMO funciona: compara os municípios líderes entre o score original
# e o score ajustado.
# ONDE é usado: validação da estratégia de priorização comercial.
# POR QUE esta escolha: permite identificar qual modelo representa
# melhor o conceito de potencial de consumo.
# QUANDO rodar: após criar score_potencial_consumo_ajustado.
# O QUE produz: ranking dos 20 municípios líderes do modelo ajustado.

df_top_20_consumo_ajustado = (
    df_municipios[
        [
            "municipio",
            "pib_municipal",
            "populacao_residente",
            "pib_per_capita",
            "score_potencial_consumo_ajustado"
        ]
    ]
    .sort_values(
        by="score_potencial_consumo_ajustado",
        ascending=False
    )
    .head(20)
)

display(df_top_20_consumo_ajustado)

,municipio,pib_municipal,populacao_residente,pib_per_capita,score_potencial_consumo_ajustado
3829,São Paulo - SP,945946483,11451999,82.600992,0.8196
3242,Rio de Janeiro - RJ,380381515,6211223,61.241001,0.3362
3215,Maricá - RJ,158394291,197277,802.902979,0.3247
5569,Brasília - DF,328789563,2817381,116.700426,0.3058
3255,Saquarema - RJ,75409090,89559,842.004600,0.2638
3677,Paulínia - SP,70496584,110537,637.764586,0.2111
3499,Ilhabela - SP,24913501,34934,713.159129,0.1905
2183,São Francisco do Conde - BA,25361789,38733,654.785041,0.1770
3155,Presidente Kennedy - ES,8179738,13696,597.235543,0.1488
3654,Osasco - SP,112145798,728615,153.916400,0.1314


## Insight de Negócio — Validação do Modelo de Potencial de Consumo

A comparação entre os modelos demonstrou que a definição dos pesos altera significativamente a interpretação do mercado.

O modelo inicial (60% PIB e 40% PIB per capita) priorizou municípios extremamente ricos por habitante, mesmo quando possuíam baixa população.

Após aumentar o peso do PIB municipal para 80%, grandes centros econômicos como Rio de Janeiro, Brasília, Curitiba, Belo Horizonte e Manaus passaram a ocupar posições de destaque, indicando maior aderência ao conceito de potencial de consumo em larga escala.

O resultado sugere que decisões comerciais baseadas exclusivamente em renda média podem superestimar mercados pequenos e subestimar regiões com elevada concentração de consumidores.

Para estratégias de expansão comercial, campanhas de marketing, abertura de unidades ou definição de territórios de vendas, o modelo ajustado apresenta maior capacidade de representar o tamanho real da oportunidade de mercado.

Conclusão analítica:

Potencial de consumo não depende apenas da riqueza individual. A combinação entre capacidade econômica e escala populacional produz uma visão mais próxima do comportamento real de mercado.